### Tiny GPT2 like Decoder only transformer with MLP

In [ ]:
#importing libraries
from pathlib import Path
import tiktoken 
import torch

#downloading the dataset
!curl.exe -L "https://github.com/karpathy/char-rnn/raw/master/data/tinyshakespeare/input.txt" -o "input.txt"
text = Path("input.txt").read_text(encoding="utf-8")

#encoding
enc = tiktoken.get_encoding("gpt2")
data = torch.tensor(enc.encode(text), dtype = torch.long)
vocab_size = enc.n_vocab

print("Data shape", data.shape, "\n Vocab size", vocab_size)
print("first 20 tokens:", data[:20].tolist())
print("\n decoded: ", enc.decode(data[:20].tolist()))

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0

  0      0   0      0   0      0      0      0                              0
100  1.06M 100  1.06M   0      0  1.17M      0                              0
100  1.06M 100  1.06M   0      0  1.17M      0                              0
100  1.06M 100  1.06M   0      0  1.17M      0                              0


Data shape torch.Size([338025]) 
 Vocab size 50257
first 20 tokens: [5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248]

 decoded:  First Citizen:
Before we proceed any further, hear me speak.

All:
Spe


## Configuring the device and sample training batches

Each batch contains input sequences and their next-token targets. The model
will use the input tokens to predict the target token at every position.

`block_size` controls how many tokens
the model sees at once

In [ ]:
# Split the token sequence: first 90% for training, final 10% for testing.
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

print("Training tokens:", len(train_data))
print("Test tokens:", len(test_data))

Training tokens: 304222
Test tokens: 33803


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

batch_size = 4
block_size = 64



def get_batch(split):
    """
    Returns a batch of randomly selected token windows.

    x contains block_size consecutive tokens.
    y contains the same windows shifted one token to the right.

    For example:
        x: [A, B, C, D]
        y: [B, C, D, E]

    Shapes will be:
        x: (batch_size, block_size)
        y: (batch_size, block_size)
    """
    source = train_data if split == "train" else test_data

    # Picking valid starting positions
    starts = torch.randint(
        low=0,
        high=len(source) - block_size,
        size=(batch_size,)
    )

    # Building input windows and the corresponding next-token labels.
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])

    # Moving batches to the same device as the model.
    return x.to(device), y.to(device)

# Inspecting a batch to confirm the dimensions and token shift.
xb, yb = get_batch("train")

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("First input tokens:", xb[0].tolist())
print("First target tokens:", yb[0].tolist())
print("Decoded input:", enc.decode(xb[0].tolist()))
print("Decoded target:", enc.decode(yb[0].tolist()))

Device: cuda
Input shape: torch.Size([4, 64])
Target shape: torch.Size([4, 64])
First input tokens: [198, 3347, 318, 1223, 878, 607, 640, 5203, 1549, 13, 198, 198, 4537, 6239, 28893, 25, 198, 32, 2933, 30, 198, 198, 3620, 4146, 3539, 25, 198, 32, 4957, 11, 290, 257, 922, 306, 42666, 11, 198, 43, 436, 88, 290, 588, 284, 2107, 25, 262, 16599, 11583, 198, 20045, 4467, 287, 470, 26, 1139, 705, 3666, 3595, 17234, 11, 198, 40, 716, 10218]
First target tokens: [3347, 318, 1223, 878, 607, 640, 5203, 1549, 13, 198, 198, 4537, 6239, 28893, 25, 198, 32, 2933, 30, 198, 198, 3620, 4146, 3539, 25, 198, 32, 4957, 11, 290, 257, 922, 306, 42666, 11, 198, 43, 436, 88, 290, 588, 284, 2107, 25, 262, 16599, 11583, 198, 20045, 4467, 287, 470, 26, 1139, 705, 3666, 3595, 17234, 11, 198, 40, 716, 10218, 355]
Decoded input: 
She is something before her time deliver'd.

PAULINA:
A boy?

EMILIA:
A daughter, and a goodly babe,
Lusty and like to live: the queen receives
Much comfort in't; says 'My poor prisoner,
I 

## Causal self-attention

Self-attention lets each token combine information from earlier tokens in its
sequence. The causal mask blocks access to future tokens: when predicting the
token after position *t*, the model must not see tokens after *t*.

Each attention head projects the input into queries, keys, and values. It
compares queries with keys to calculate attention weights, applies the causal
mask, then uses those weights to combine values.

In [ ]:
class Head(nn.Module):
    #One causal self-attention head

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()

        # Learnable projections from each token representation to keys,
        # queries, and values.
        self.key = nn.Linear(n_embd, head_size, bias=False) #bias is not implemented, inline with the original transformer paper
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # A lower-triangular mask permits attention to the current and
        # preceding positions, while blocking all future positions.
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x has shape: (batch, time, embedding_width)
        batch, time, channels = x.shape

        key = self.key(x)       # (batch, time, head_size)
        query = self.query(x)   # (batch, time, head_size)
        value = self.value(x)   # (batch, time, head_size)

        # Compare every query with every key. Dividing by sqrt(head_size)
        # This keeps dot products from growing too large as dimensions increase.
        scores = query @ key.transpose(-2, -1)
        scores = scores * (key.shape[-1] ** -0.5)

        # Replace future-position scores with negative infinity. 
        scores = scores.masked_fill(
            self.causal_mask[:time, :time] == 0,
            float("-inf")
        )

        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Combine the value vectors using the attention probabilities.
        return weights @ value

## Multi-head attention and feed-forward layer

Multi-head attention runs several attention heads in parallel. Each head can
learn to focus on different relationships between tokens. Their outputs are
joined and projected back to the model's embedding width.

The feed-forward layer applies a same small neural network for predictions.

In [ ]:
class MultiHeadAttention(nn.Module):
    #Running several causal attention heads in parallel

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()

        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head.")

        head_size = n_embd // n_head

        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout)
            for _ in range(n_head)
        ])

        # Combine the heads and map their concatenated output back to n_embd.
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Each head returns (batch, time, head_size). Concatenate on the
        # feature dimension to get (batch, time, n_embd).
        joined = torch.cat([head(x) for head in self.heads], dim=-1)

        return self.dropout(self.projection(joined))

In [ ]:
class FeedForward(nn.Module):
    #A position-wise MLP that processes each token representation

    def __init__(self, n_embd, dropout):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),

            # ReLU adds a non-linear transformation.
            nn.ReLU(),

            # Project back to the original embedding width.
            nn.Linear(4 * n_embd, n_embd),

            # Dropout regularizes training by randomly zeroing some activations.
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.network(x)

## Defining Decoder block

Decoder block applies causal self-attention and then a feed-forward layer.
Each sublayer uses a residual connection, which adds its input back to its
output. Layer normalization is applied before each sublayer.

In [ ]:
class DecoderBlock(nn.Module):
    #One Transformer decoder block with pre-layer normalization

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()

        self.attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            block_size=block_size,
            dropout=dropout
        )
        self.feed_forward = FeedForward(n_embd, dropout)

        self.norm1 = nn.LayerNorm(n_embd)
        self.norm2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Normalize, apply attention, and add the original input back.
        x = x + self.attention(self.norm1(x))

        # Normalize, apply the MLP, and add that sublayer's input back too.
        x = x + self.feed_forward(self.norm2(x))

        return x

## Assembling the decoder-only Transformer

The model adds a learned token embedding and a learned position embedding.
The combined representations pass through a stack of decoder blocks.


In [ ]:
class DecoderTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        n_embd=128,
        n_head=4,
        n_layer=10,
        block_size=64,
        dropout=0.1
    ):
        super().__init__()

        self.block_size = block_size

        # Converting token IDs and positions into learned vectors.
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)

        # Stacking decoder blocks so each layer can build on earlier processing.
        self.blocks = nn.Sequential(*[
            DecoderBlock(n_embd, n_head, block_size, dropout)
            for _ in range(n_layer)
        ])

        self.final_norm = nn.LayerNorm(n_embd)

        # Mapping each final token representation to scores over the vocabulary.
        self.language_model_head = nn.Linear(n_embd, vocab_size)

    def forward(self, token_ids, targets=None):
        batch, time = token_ids.shape

        if time > self.block_size:
            raise ValueError(
                f"Sequence length {time} exceeds block_size {self.block_size}."
            )

        # Embedding token identity and token position, then adding the vectors.
        positions = torch.arange(time, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)

        # Processing the sequence with the decoder layers.
        x = self.blocks(x)
        x = self.final_norm(x)

        # Producing next-token logits for every position.
        logits = self.language_model_head(x)

        loss = None
        if targets is not None:
            # Cross entropy expects a 2-D array of logits and a 1-D array of
            # target IDs, so combining batch and time dimensions.
            loss = F.cross_entropy(
                logits.reshape(batch * time, vocab_size),
                targets.reshape(batch * time)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, token_ids, max_new_tokens, temperature=1.0, top_k=None):
        """Sample new tokens autoregressively from the model."""
        if temperature <= 0:
            raise ValueError("temperature must be greater than zero.")

        for _ in range(max_new_tokens):
            # The model can use at most block_size tokens as context.
            context = token_ids[:, -self.block_size :]

            # Only the final position's logits are needed for the next token.
            logits, _ = self(context)
            next_logits = logits[:, -1, :] / temperature

            # Optionally restricting sampling to the top_k highest-scoring tokens.
            if top_k is not None:
                k = min(top_k, next_logits.shape[-1])
                threshold = torch.topk(next_logits, k).values[:, [-1]]
                next_logits = next_logits.masked_fill(
                    next_logits < threshold,
                    float("-inf")
                )

            probabilities = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)

            # Appending the sampled token to the sequence and using it as context
            # for the next generation step.
            token_ids = torch.cat([token_ids, next_token], dim=1)

        return token_ids

## Creating a model instance and optimizer

This configuration is deliberately small, suitable for an RTX 4060. GPT-2 tokenization also has a vocabulary
of about 50,000 tokens. If we run out of memory, then we can lower `batch_size`, `n_embd`, or `n_layer`.

In [ ]:
# Keeping the batch sampler and model context length in agreement.
assert block_size == 64

model = DecoderTransformer(
    vocab_size=vocab_size,
    n_embd=128,
    n_head=4,
    n_layer=4,
    block_size=block_size,
    dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

num_parameters = sum(p.numel() for p in model.parameters())
print(f"Number of model parameters: {num_parameters}")

Number of model parameters: 13716049


## Estimating training and test loss

The loss is measured on batches sampled from each split. Evaluation mode
disables dropout, and `torch.no_grad()` avoids storing gradients because
evaluation should not update the model.

In [ ]:
@torch.no_grad()
def estimate_loss(eval_batches=20):
    #Returns average loss estimates for the train and test splits
    model.eval()
    results = {}

    for split in ("train", "test"):
        losses = torch.zeros(eval_batches)

        for batch_index in range(eval_batches):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[batch_index] = loss.item()

        results[split] = losses.mean().item()

    # Restoring training mode so dropout is active during optimization.
    model.train()
    return results

## Training the model

Every training step samples a batch, computes next-token loss, calculates
gradients with backpropagation, and updates the model parameters.


In [20]:
max_iters = 10000
eval_interval = 100
learning_rate = 3e-4

model.train()

for step in range(max_iters):
    # Periodically checking whether the model is improving on both data splits.
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(
            f"step {step:4d} | "
            f"train loss: {losses['train']:.4f} | "
            f"test loss: {losses['test']:.4f}"
        )

    x, y = get_batch("train")

    # Forward pass: calculate predictions and the loss.
    _, loss = model(x, y)

    # Clearing old gradients, computing gradients for this batch, and update weights.
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step    0 | train loss: 2.5912 | test loss: 5.3278
step  100 | train loss: 2.5692 | test loss: 5.6437
step  200 | train loss: 2.5739 | test loss: 5.4296
step  300 | train loss: 2.5735 | test loss: 5.4967
step  400 | train loss: 2.5909 | test loss: 5.3438
step  500 | train loss: 2.4897 | test loss: 5.5301
step  600 | train loss: 2.3877 | test loss: 5.4730
step  700 | train loss: 2.4535 | test loss: 5.5751
step  800 | train loss: 2.5446 | test loss: 5.7463
step  900 | train loss: 2.5477 | test loss: 5.0935
step 1000 | train loss: 2.5538 | test loss: 5.4583
step 1100 | train loss: 2.5120 | test loss: 5.6492
step 1200 | train loss: 2.4506 | test loss: 5.7894
step 1300 | train loss: 2.4645 | test loss: 5.3959
step 1400 | train loss: 2.5164 | test loss: 5.4077
step 1500 | train loss: 2.4808 | test loss: 5.6538
step 1600 | train loss: 2.4858 | test loss: 5.4650
step 1700 | train loss: 2.4608 | test loss: 5.5271
step 1800 | train loss: 2.4992 | test loss: 5.5589
step 1900 | train loss: 2.4784 

## Generating the text

Generation starts from a prompt tokenized with the same GPT-2 encoding used
for training. The model samples one next token at a time and appends it to the
context.

`temperature` controls randomness. Lower values favor more likely tokens;
higher values make sampling more varied. `top_k` optionally limits sampling
to the highest-scoring tokens.

In [18]:
model.eval()

prompt = "ROMEO:"
prompt_ids = torch.tensor(
    [enc.encode(prompt)],
    dtype=torch.long,
    device=device
)

generated_ids = model.generate(
    prompt_ids,
    max_new_tokens=300,
    temperature=0.8,
    top_k=50
)

print(enc.decode(generated_ids[0].tolist()))

ROMEO:
Thou wouldst have me stiff in the heat.

MERCUTIO:
No, no, no; here he had been commanded,
That Ancus chinks, nor on mine.

ROMEO:
No matter.

MERCUTIO:
Is whispering sir, so I swear, to-night
Will to make my fortune's friend, or to die,
Which love I have done thee,
And not my love to my mother's good,
To speak my brother's grave and brothers,
And I'll renew the villain of my short date
Will not be avoided fair as if I,
As my fortune in myself could do;
And for that I shall not be happy by.

KING LEWIS XI:
Why, then, what way were this?
Is our king? how doth deny what you do,
It is, that was your former queen?

QUEEN ELIZABETH:
Kind, as the greatest thing I can refrain of?

KING RICHARD III:
I took the name, and not to have you?

Messenger:
Now, in plainest fault! and what he were
In accustom'd; and when he shutslips,
To think it is the cause of the people,
A most gib shepherdess'd him to his crown,
And he could his noble king, how

